In [0]:
sales_path='development_042_silver_sandbox.demand_forecast.sales_silver'
like_games_path='development_042_silver_sandbox.demand_forecast.clustering_core3_like_games'
time_series_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'
sales_cl='expr1_sum'
sales_cl_cumsum='cumsum_expr1_sum'
performance=['avg_coin_in_index_vs_house','avg_theo_net_win_index_vs_house']
weight_col=['no_of_slots']
result_dest='development_042_silver_sandbox.demand_forecast.time_series_extrapolated'
performance_dest='development_042_silver_sandbox.demand_forecast.model_performance'

In [0]:
from pyspark.sql import functions as F
like_games_all = spark.table(like_games_path)

In [0]:
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import functions as F

# ============================================================
# Configuration
# ============================================================
EXPERIMENT_COL = "experiment_name"
TARGET_COL = "target_game_name"
DISTANCE_COL = "cosine_distance"

MIN_LIKE_GAMES = 3
REQUIRED_PAIR_COVERAGE_PCT = 90.0
NUMBER_OF_THRESHOLDS = 101


# ============================================================
# Prepare valid data
# ============================================================
distance_df = (
    like_games_all
    .select(
        F.col(EXPERIMENT_COL),
        F.col(TARGET_COL),
        F.col(DISTANCE_COL).cast("double").alias(DISTANCE_COL),
    )
    .filter(
        F.col(EXPERIMENT_COL).isNotNull()
        & F.col(TARGET_COL).isNotNull()
        & F.col(DISTANCE_COL).isNotNull()
        & ~F.isnan(F.col(DISTANCE_COL))
    )
)

total_rows = distance_df.count()

total_pairs = (
    distance_df
    .select(EXPERIMENT_COL, TARGET_COL)
    .distinct()
    .count()
)

if total_rows == 0:
    raise ValueError("No valid cosine-distance rows were found.")

if total_pairs == 0:
    raise ValueError(
        "No valid experiment and target-game pairs were found."
    )


# ============================================================
# Check how many rows each experiment/game pair has
# ============================================================
pair_counts = (
    distance_df
    .groupBy(EXPERIMENT_COL, TARGET_COL)
    .agg(F.count("*").alias("number_of_like_games"))
)

print("Number of rows per experiment/game pair:")

display(
    pair_counts
    .groupBy("number_of_like_games")
    .count()
    .orderBy("number_of_like_games")
)


# ============================================================
# Create candidate thresholds
# ============================================================
distance_limits = distance_df.agg(
    F.min(DISTANCE_COL).alias("min_distance"),
    F.max(DISTANCE_COL).alias("max_distance"),
).first()

min_distance = float(distance_limits["min_distance"])
max_distance = float(distance_limits["max_distance"])

if min_distance == max_distance:
    threshold_values = [min_distance]
else:
    threshold_values = np.linspace(
        min_distance,
        max_distance,
        NUMBER_OF_THRESHOLDS,
    ).tolist()

threshold_df = spark.createDataFrame(
    [(float(value),) for value in threshold_values],
    ["threshold"],
)


# ============================================================
# Count retained like games per experiment/game pair
# ============================================================
retained_by_pair = (
    F.broadcast(threshold_df)
    .crossJoin(distance_df)
    .filter(F.col(DISTANCE_COL) <= F.col("threshold"))
    .groupBy(
        "threshold",
        EXPERIMENT_COL,
        TARGET_COL,
    )
    .agg(
        F.count("*").alias("like_games_retained")
    )
)


# ============================================================
# Summarize each threshold
# ============================================================
threshold_summary = (
    retained_by_pair
    .groupBy("threshold")
    .agg(
        F.sum("like_games_retained").alias("rows_retained"),

        F.count("*").alias(
            "pairs_with_at_least_one"
        ),

        F.sum(
            F.when(
                F.col("like_games_retained") >= MIN_LIKE_GAMES,
                1,
            ).otherwise(0)
        ).alias("pairs_with_minimum"),
    )
    .withColumn(
        "rows_retained_pct",
        F.col("rows_retained")
        / F.lit(total_rows)
        * 100,
    )
    .withColumn(
        "pairs_with_at_least_one_pct",
        F.col("pairs_with_at_least_one")
        / F.lit(total_pairs)
        * 100,
    )
    .withColumn(
        "pairs_with_minimum_pct",
        F.col("pairs_with_minimum")
        / F.lit(total_pairs)
        * 100,
    )
    .withColumn(
        "average_like_games_per_pair",
        F.col("rows_retained")
        / F.lit(total_pairs),
    )
    .orderBy("threshold")
)

threshold_summary_pd = threshold_summary.toPandas()


# ============================================================
# Select the recommended threshold
# ============================================================
valid_thresholds = threshold_summary_pd[
    threshold_summary_pd["pairs_with_minimum_pct"]
    >= REQUIRED_PAIR_COVERAGE_PCT
]

if not valid_thresholds.empty:
    recommended_threshold = float(
        valid_thresholds.iloc[0]["threshold"]
    )

    selection_reason = (
        f"Smallest threshold where at least "
        f"{REQUIRED_PAIR_COVERAGE_PCT:.0f}% of experiment/game pairs "
        f"retain at least {MIN_LIKE_GAMES} like games."
    )

else:
    best_row_index = (
        threshold_summary_pd["pairs_with_minimum_pct"].idxmax()
    )

    recommended_threshold = float(
        threshold_summary_pd.loc[
            best_row_index,
            "threshold",
        ]
    )

    selection_reason = (
        f"No threshold gave "
        f"{REQUIRED_PAIR_COVERAGE_PCT:.0f}% coverage. "
        f"Using the threshold with the highest available coverage."
    )


# ============================================================
# Plot threshold tradeoff
# ============================================================
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 10),
    sharex=True,
)

axes[0].plot(
    threshold_summary_pd["threshold"],
    threshold_summary_pd["rows_retained_pct"],
    linewidth=2.5,
    label="All like-game rows retained",
)

axes[0].plot(
    threshold_summary_pd["threshold"],
    threshold_summary_pd["pairs_with_at_least_one_pct"],
    linewidth=2.5,
    label="Experiment/game pairs retaining at least 1",
)

axes[0].plot(
    threshold_summary_pd["threshold"],
    threshold_summary_pd["pairs_with_minimum_pct"],
    linewidth=2.5,
    label=(
        f"Experiment/game pairs retaining at least "
        f"{MIN_LIKE_GAMES}"
    ),
)

axes[0].axhline(
    REQUIRED_PAIR_COVERAGE_PCT,
    color="gray",
    linestyle=":",
    label=(
        f"Desired coverage: "
        f"{REQUIRED_PAIR_COVERAGE_PCT:.0f}%"
    ),
)

axes[0].axvline(
    recommended_threshold,
    color="red",
    linestyle="--",
    label=(
        f"Recommended threshold: "
        f"{recommended_threshold:.4f}"
    ),
)

axes[0].set_ylabel("Percentage")
axes[0].set_ylim(0, 105)
axes[0].set_title(
    "Cosine-distance threshold tradeoff"
)
axes[0].grid(alpha=0.25)
axes[0].legend()


# ============================================================
# Average retained per experiment/game pair
# ============================================================
axes[1].plot(
    threshold_summary_pd["threshold"],
    threshold_summary_pd[
        "average_like_games_per_pair"
    ],
    linewidth=2.5,
    color="purple",
)

axes[1].axvline(
    recommended_threshold,
    color="red",
    linestyle="--",
)

axes[1].axhline(
    MIN_LIKE_GAMES,
    color="gray",
    linestyle=":",
)

axes[1].set_xlabel(
    "Maximum cosine distance allowed"
)
axes[1].set_ylabel(
    "Average like games retained"
)
axes[1].set_title(
    "Average retained per experiment/game pair"
)
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


# ============================================================
# Filter the original DataFrame
# ============================================================
like_games_filtered = (
    like_games_all
    .filter(
        F.col(DISTANCE_COL).cast("double")
        <= F.lit(recommended_threshold)
    )
)

filtered_pair_summary = (
    like_games_filtered
    .groupBy(EXPERIMENT_COL, TARGET_COL)
    .agg(
        F.count("*").alias("like_games_retained"),
        F.max(DISTANCE_COL).alias(
            "largest_retained_distance"
        ),
    )
)


# ============================================================
# Results
# ============================================================
print(
    f"Recommended threshold: "
    f"{recommended_threshold:.6f}"
)
print(selection_reason)
print(f"Total experiment/game pairs: {total_pairs:,}")
print(f"Rows before filtering: {total_rows:,}")
print(f"Rows after filtering: {like_games_filtered.count():,}")

display(threshold_summary)

display(
    filtered_pair_summary.orderBy(
        EXPERIMENT_COL,
        TARGET_COL,
    )
)

display(
    like_games_filtered.orderBy(
        EXPERIMENT_COL,
        TARGET_COL,
        F.col(DISTANCE_COL).asc(),
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# Databricks widgets
# ============================================================
dbutils.widgets.text(
    "cosine_distance_threshold",
    "0.30",
    "Maximum cosine distance",
)

dbutils.widgets.dropdown(
    "minimum_like_games",
    "3",
    ["1", "2", "3", "4", "5"],
    "Minimum like games per target",
)


# ============================================================
# Read and validate widget values
# ============================================================
cosine_distance_threshold = float(
    dbutils.widgets.get("cosine_distance_threshold")
)

minimum_like_games = int(
    dbutils.widgets.get("minimum_like_games")
)

if not 0 <= cosine_distance_threshold <= 2:
    raise ValueError(
        "cosine_distance_threshold must be between 0 and 2."
    )

if minimum_like_games < 1:
    raise ValueError(
        "minimum_like_games must be at least 1."
    )

print(
    f"Cosine-distance threshold: "
    f"{cosine_distance_threshold}"
)

print(
    f"Minimum like games per experiment/target: "
    f"{minimum_like_games}"
)


# ============================================================
# Rank games within each experiment and target game
# Lower cosine distance is better
# ============================================================
ranking_window = (
    Window
    .partitionBy(
        "experiment_name",
        "target_game_name",
    )
    .orderBy(
        F.col("cosine_distance").asc_nulls_last()
    )
)

like_games_ranked = (
    like_games_all
    .withColumn(
        "_cosine_distance",
        F.col("cosine_distance").cast("double"),
    )
    .withColumn(
        "distance_rank",
        F.row_number().over(ranking_window),
    )
    .withColumn(
        "passes_threshold",
        (
            F.col("_cosine_distance").isNotNull()
            & (
                F.col("_cosine_distance")
                <= F.lit(cosine_distance_threshold)
            )
        ),
    )
)


# ============================================================
# Apply threshold and minimum-game fallback
#
# A row is retained when:
# 1. It passes the threshold, OR
# 2. It is one of the closest X games
#
# This guarantees at least X like games when X rows are available.
# ============================================================
like_games_filtered = (
    like_games_ranked
    .filter(
        F.col("passes_threshold")
        | (
            F.col("distance_rank")
            <= F.lit(minimum_like_games)
        )
    )
    .withColumn(
        "retained_reason",
        F.when(
            F.col("passes_threshold"),
            F.lit("passed_threshold"),
        ).otherwise(
            F.lit("minimum_game_fallback")
        ),
    )
    .drop("_cosine_distance")
)


# ============================================================
# Summary per experiment and target game
# ============================================================
filter_summary = (
    like_games_filtered
    .groupBy(
        "experiment_name",
        "target_game_name",
    )
    .agg(
        F.count("*").alias("like_games_retained"),

        F.sum(
            F.when(
                F.col("retained_reason")
                == "passed_threshold",
                1,
            ).otherwise(0)
        ).alias("games_passing_threshold"),

        F.sum(
            F.when(
                F.col("retained_reason")
                == "minimum_game_fallback",
                1,
            ).otherwise(0)
        ).alias("games_added_by_fallback"),

        F.min("cosine_distance").alias(
            "best_cosine_distance"
        ),

        F.max("cosine_distance").alias(
            "worst_retained_cosine_distance"
        ),
    )
)


# ============================================================
# Display results
# ============================================================
display(
    filter_summary.orderBy(
        "experiment_name",
        "target_game_name",
    )
)

display(
    like_games_filtered.orderBy(
        "experiment_name",
        "target_game_name",
        "distance_rank",
    )
)
like_games_all=like_games_filtered

In [0]:
from pyspark.sql import functions as F

AVAILABLE_ANALYSIS_SPLITS = ["valid", "test"]
dbutils.widgets.dropdown(
    "analysis_split",
    "valid",
    AVAILABLE_ANALYSIS_SPLITS,
    "Like-game target split",
)
analysis_split = dbutils.widgets.get("analysis_split")

sales_df = spark.table(sales_path)
time_series_df = spark.table(time_series_path).filter("own_status = 'owned'")

# Discover all run_ids that have valid data in the selected split
#like_games_all = spark.table(like_games_path)



# Filter like-games to selected run_id + split
like_games_df =like_games_all

display(sales_df)
display(time_series_df)


In [0]:
#the contribution of each like game to the target game's performance is calculated as follows:
#the cosine distance max is used for each game-experiment pair is used to noramalize for pairs 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

group_window = Window.partitionBy(
    "target_game_name",
    "mlflow_run_id",
)
like_games_all = (
    like_games_all
    .withColumn(
        "_max_cosine_distance",
        F.max("cosine_distance").over(group_window),
    )
    .withColumn(
        "rank_weight",
        F.when(
            F.col("_max_cosine_distance") > 0,
             0.1 + 0.9 * (1 - (
                F.col("cosine_distance") /
                F.col("_max_cosine_distance")
            )),
        ).otherwise(F.lit(1.0)),
    )
    .drop("_max_cosine_distance")
)
display(like_games_all)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Configuration ---
weight_col_name = weight_col[0]        # 'no_of_slots'
metric_cols = performance + [sales_cl]  # performance indices + sales
curve_length_method = 'min'             # how to determine curve length: 'min', 'max', 'avg'
max_curve_length = 12

# --- Norm weights for combining weighting factors ---
# Dict of {column_name: exponent}.  Each factor is raised to its exponent
# before multiplication.  1.0 = linear; >1 amplifies; <1 dampens.
# Values are clipped to min 1 before exponentiation to avoid zero/negative issues.
norm_weights = {
    "no_of_slots": 1.1,              # slot count weight
    "rank_weight": 1.1,              # cosine distance similarity weight
    "months_since_release": 0.5,     # game maturity weight
}
# Extra norm columns to carry through the pipeline (weight_col & rank_weight
# are already handled separately in selects)
_extra_norm_cols = [c for c in norm_weights if c not in (weight_col_name, "rank_weight")]
# Backward compat for downstream cells (cell 9)
slots_norm_weight = norm_weights["no_of_slots"]
cosine_norm_weight = norm_weights["rank_weight"]

# --- Step 1: Build per-like-game monthly time series ---
# Get mapping: target_game_name -> game_name (each target game has ~3 like games)
mapping_df = like_games_all.select("target_game_name", "game_name", "neighbor_rank",'experiment_name','mlflow_run_id', 'rank_weight').dropDuplicates(["target_game_name", "game_name"])

# Get performance time series for like games (game_name, yearmonth, performance, weight)
perf_df = time_series_df.select(
    F.col("game_name"),
    F.col("yearmonth"),
    F.col("own_status"),
    F.col(weight_col_name).cast("double").alias(weight_col_name),
    *[F.col(c).cast("double").alias(c) for c in _extra_norm_cols],
    *[F.col(c).cast("double").alias(c) for c in performance]
)

# Get sales for like games (ep_theme_name, beginning_month_date, expr1_sum)
# Deduplicate sales to one record per (game, month, revenue_type)
unique_sales = sales_df.dropDuplicates(["beginning_month_date", "revenue_type", sales_cl, "ep_theme_name"])
sales_agg = unique_sales.groupBy("ep_theme_name", "beginning_month_date").agg(
    F.sum(F.col(sales_cl).cast("double")).alias(sales_cl)
)

# --- Step 2: Join performance and sales for each like game on date ---
# Outer join so performance months without sales get 0 and we keep full curves
game_ts = perf_df.join(
    sales_agg,
    (perf_df.game_name == sales_agg.ep_theme_name) & (perf_df.yearmonth == sales_agg.beginning_month_date),
    how="left"
).select(
    perf_df.game_name,
    perf_df.yearmonth,
    F.col("own_status"),
    F.col(weight_col_name),
    *[F.col(c) for c in _extra_norm_cols],
    *[F.col(c) for c in performance],
    F.coalesce(F.col(sales_cl), F.lit(0.0)).alias(sales_cl)
)

# --- Step 3: Map to target games ---
train_data = mapping_df.join(
    game_ts,
    mapping_df.game_name == game_ts.game_name,
    "inner"
).select(
    "target_game_name",
    mapping_df.game_name,
    "neighbor_rank",
    "yearmonth",
    'experiment_name',
    'mlflow_run_id',
    "own_status",
    weight_col_name,
    "rank_weight",
    *_extra_norm_cols,
    *[F.col(c) for c in performance],
    F.col(sales_cl)
)
# --- Compute combined weight from configurable norm_weights dict ---
_cw_expr = F.lit(1.0)
for _col_name, _exp in norm_weights.items():
    _cw_expr = _cw_expr * F.pow(
        F.greatest(F.col(_col_name), F.lit(1.0)), F.lit(_exp)
    )
train_data = train_data.withColumn("combined_weight", _cw_expr)
# --- Step 4: Assign ordinal month index (removes calendar alignment) ---
w = Window.partitionBy("target_game_name", "game_name").orderBy("yearmonth")
train_data = train_data.withColumn("month_index", F.row_number().over(w))

# --- Step 4b: Compute weighted std from ALL like games (before zero-sales filter) ---
# This ensures bands are meaningful even when only 1 like game survives the sales filter
_sw = F.sum(F.col("combined_weight"))
unfiltered_std = train_data.groupBy("target_game_name", "month_index").agg(
    *[
        F.when(_sw > 0,
               F.sqrt(F.greatest(
                   F.sum(F.col(c) * F.col(c) * F.col("combined_weight")) / _sw -
                   F.pow(F.sum(F.col(c) * F.col("combined_weight")) / _sw, 2),
                   F.lit(0.0)
               ))
        ).otherwise(F.lit(0.0)).alias(f"{c}_std")
        for c in metric_cols
    ]
)

# --- Step 5: Exclude like games with zero total sales (no signal) ---
total_sales = train_data.groupBy("target_game_name", "game_name").agg(
    F.sum(sales_cl).alias("total_sales")
)
valid_games = total_sales.filter(F.col("total_sales") > 0).select("target_game_name", "game_name")
train_data = train_data.join(valid_games, ["target_game_name", "game_name"], "inner")

# --- Step 6: Determine curve length per target game ---
game_lengths = train_data.groupBy("target_game_name", "game_name").agg(
    F.max("month_index").alias("game_length")
)
if curve_length_method == 'min':
    curve_lengths = game_lengths.groupBy("target_game_name").agg(F.min("game_length").alias("curve_length"))
elif curve_length_method == 'max':
    curve_lengths = game_lengths.groupBy("target_game_name").agg(F.max("game_length").alias("curve_length"))
else:
    curve_lengths = game_lengths.groupBy("target_game_name").agg(
        F.round(F.avg("game_length")).cast("int").alias("curve_length")
    )
curve_lengths = curve_lengths.withColumn(
    "curve_length",
    F.least(F.col("curve_length"), F.lit(max_curve_length))
)

# Trim to curve length
train_data = train_data.join(curve_lengths, "target_game_name", "inner")
train_data = train_data.filter(F.col("month_index") <= F.col("curve_length"))

# --- Step 7: Weighted aggregation across like games per ordinal month ---
sum_weight = F.sum("combined_weight")
agg_exprs = [
    F.when(sum_weight > 0, F.sum(F.col(c) * F.col("combined_weight")) / sum_weight)
     .otherwise(0.0).alias(c)
    for c in metric_cols
]
agg_exprs.append(sum_weight.alias("total_combined_weight"))
agg_exprs.append(F.countDistinct("game_name").alias("n_like_games_used"))
agg_exprs.append(F.sort_array(F.collect_set("game_name")).alias("like_games_used"))

predicted_curve = train_data.groupBy("target_game_name",'mlflow_run_id', 'experiment_name',"month_index", "curve_length", "own_status").agg(*agg_exprs)
predicted_curve = predicted_curve.orderBy("target_game_name", "month_index")
predicted_curve = predicted_curve.withColumns({
    "dataset_split": F.lit(analysis_split),
    "cosine_distance_threshold": F.lit(cosine_distance_threshold),
    "norm_weights": F.lit(str(norm_weights)),
})

# --- Step 8: Cumulative sum of sales over the curve ---
w_cum = Window.partitionBy("target_game_name").orderBy("month_index")
predicted_curve = predicted_curve.withColumn(f"cumsum_{sales_cl}", F.sum(sales_cl).over(w_cum))

# --- Step 9: Join unfiltered std (from ALL like games) and compute ±2σ bands ---
predicted_curve = predicted_curve.join(unfiltered_std, ["target_game_name", "month_index"], "left")
for c in metric_cols:
    predicted_curve = predicted_curve.withColumn(f"{c}_upper", F.col(c) + 2 * F.col(f"{c}_std"))
    predicted_curve = predicted_curve.withColumn(f"{c}_lower", F.col(c) - 2 * F.col(f"{c}_std"))

# Cumulative sales bands (cumulate upper/lower separately)
predicted_curve = predicted_curve.withColumn(f"cumsum_{sales_cl}_upper", F.sum(F.col(f"{sales_cl}_upper")).over(w_cum))
predicted_curve = predicted_curve.withColumn(f"cumsum_{sales_cl}_lower", F.sum(F.col(f"{sales_cl}_lower")).over(w_cum))

print(f"Curve length method: {curve_length_method} | Max curve length: {max_curve_length}")
display(predicted_curve)

In [0]:
display(train_data)

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# Process ALL (run_id, dataset_split) combinations and save
# ============================================================

# Load all like-game data without filtering to one run.
like_games_all_df = like_games_all #spark.table(like_games_path)

# Get all distinct run, split, and experiment combinations.
all_combos = (
    like_games_all_df
    .select(
        F.col("mlflow_run_id").alias("run_id"),
        "dataset_split",
        "experiment_name",
    )
    .distinct()
    .collect()
)

print(
    f"Processing {len(all_combos)} "
    "(run_id, dataset_split, experiment_name) combinations..."
)


# ============================================================
# Configuration
# ============================================================

shift_flag_batch = (
    dbutils.widgets.get("shift_flag")
    .strip()
    .lower()
    == "true"
)

weight_col_name = weight_col[0] # no of slots

curve_length_method_batch = "min" # if two games have difrentiating lenghts 
max_curve_length_batch = 12

# Norm weights (norm_weights dict) and cosine threshold — defined in cell 7 config
# norm_weights, _extra_norm_cols, cosine_distance_threshold


# ============================================================
# Helper functions
# ============================================================

def _fit_linear(x, y):
    denominator = np.dot(x, x)

    if denominator == 0:
        a = 0.0
    else:
        a = np.dot(x, y) / denominator

    return np.array([a]), lambda t: a * t


def _fit_poly2(x, y):
    X = np.column_stack([x, x**2])

    coeffs, _, _, _ = np.linalg.lstsq(
        X,
        y,
        rcond=None,
    )

    return (
        coeffs,
        lambda t: coeffs[0] * t + coeffs[1] * t**2,
    )


def _fit_log(x, y):
    log_x = np.log(x + 1)
    denominator = np.dot(log_x, log_x)

    if denominator == 0:
        a = 0.0
    else:
        a = np.dot(log_x, y) / denominator

    return (
        np.array([a]),
        lambda t: a * np.log(t + 1),
    )


def _r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum(
        (y_true - np.mean(y_true)) ** 2
    )

    if ss_tot == 0:
        return 0.0

    return 1.0 - ss_res / ss_tot


def _clip_nonnegative(values):
    """
    Clip negative values to zero without shifting
    the entire curve upward.
    """
    return np.maximum(
        np.asarray(values, dtype=float),
        0.0,
    )


def _is_all_zero(values): # checks if all 0s or null 
    values = (
        values
        .dropna()
        .to_numpy(dtype=float)
    )

    return (
        len(values) > 0
        and np.all(values == 0)
    )


# ============================================================
# Pre-compute shared sales data
# ============================================================

unique_sales = sales_df.dropDuplicates(
    [
        "beginning_month_date",
        "revenue_type",
        sales_cl,
        "ep_theme_name",
    ]
)

sales_agg_batch = (
    unique_sales
    .groupBy(
        "ep_theme_name",
        "beginning_month_date",
    )
    .agg(
        F.sum(
            F.col(sales_cl).cast("double")
        ).alias(sales_cl)
    )
)


# ============================================================
# Pre-compute shared performance data
# ============================================================

perf_df_batch = time_series_df.select(
    F.col("game_name"),
    F.col("yearmonth"),
    F.col("own_status"),
    F.col(weight_col_name)
    .cast("double")
    .alias(weight_col_name),
    *[
        F.col(c).cast("double").alias(c)
        for c in _extra_norm_cols
    ],
    *[
        F.col(c).cast("double").alias(c)
        for c in performance
    ],
)


# ============================================================
# Pre-compute all actual performance data
# ============================================================

actual_perf_window = (
    Window
    .partitionBy("game_name")
    .orderBy("yearmonth")
)

all_actual_perf_spark = (
    time_series_df
    .withColumn(
        "release_month",
        F.row_number().over(actual_perf_window),
    )
    .select(
        F.col("game_name").alias("target_game_name"),
        "release_month",
        *[
            F.col(c)
            .cast("double")
            .alias(f"actual_{c}")
            for c in performance
        ],
    )
)

all_actual_perf_pd = (
    all_actual_perf_spark
    .toPandas()
)


# ============================================================
# Pre-compute all actual sales data
# ============================================================

actual_sales_window = (
    Window
    .partitionBy("ep_theme_name")
    .orderBy("beginning_month_date")
)

all_actual_sales_spark = (
    unique_sales
    .groupBy(
        "ep_theme_name",
        "beginning_month_date",
    )
    .agg(
        F.sum(
            F.col(sales_cl).cast("double")
        ).alias(f"actual_{sales_cl}")
    )
    .withColumn(
        "release_month",
        F.row_number().over(actual_sales_window),
    )
    .select(
        F.col("ep_theme_name").alias(
            "target_game_name"
        ),
        "release_month",
        f"actual_{sales_cl}",
    )
)

all_actual_sales_pd = (
    all_actual_sales_spark
    .toPandas()
)


# ============================================================
# Combine all actual performance and sales
# ============================================================

all_actuals_pd = (
    all_actual_perf_pd
    .merge(
        all_actual_sales_pd,
        on=[
            "target_game_name",
            "release_month",
        ],
        how="outer",
    )
    .sort_values(
        [
            "target_game_name",
            "release_month",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# Main loop
# ============================================================

all_forecast_results = []
all_model_summaries = []

for combo in all_combos:

    current_run_id = combo["run_id"]
    current_exp_name = combo["experiment_name"]
    current_split = combo["dataset_split"]

    print(
        f"  → run_id={current_run_id[:12]}..., "
        f"split={current_split}, "
        f"experiment={current_exp_name}"
    )

    like_games_filtered = (
        like_games_all_df
        .filter(
            F.col("mlflow_run_id")
            == current_run_id
        )
        .filter(
            F.col("dataset_split")
            == current_split
        )
        .filter(
            F.col("experiment_name")
            == current_exp_name
        )
    )

    if like_games_filtered.limit(1).count() == 0:
        continue


    # ========================================================
    # Stage 1: Prepare like-game mapping
    # ========================================================

    mapping_df = (
        like_games_filtered
        .select(
            "target_game_name",
            "game_name",
            "neighbor_rank",
            "rank_weight",
        )
        .dropDuplicates(
            [
                "target_game_name",
                "game_name",
            ]
        )
    )


    # ========================================================
    # Join performance and sales
    # ========================================================

    game_ts = (
        perf_df_batch
        .join(
            sales_agg_batch,
            (
                perf_df_batch.game_name
                == sales_agg_batch.ep_theme_name
            )
            & (
                perf_df_batch.yearmonth
                == sales_agg_batch.beginning_month_date
            ),
            how="left",
        )
        .select(
            perf_df_batch.game_name,
            perf_df_batch.yearmonth,
            F.col("own_status"),
            F.col(weight_col_name),
            *[
                F.col(c)
                for c in _extra_norm_cols
            ],
            *[
                F.col(c)
                for c in performance
            ],
            F.coalesce(
                F.col(sales_cl),
                F.lit(0.0),
            ).alias(sales_cl),
        )
    )

    train_data = (
        mapping_df
        .join(
            game_ts,
            mapping_df.game_name
            == game_ts.game_name,
            "inner",
        )
        .select(
            "target_game_name",
            mapping_df.game_name,
            "neighbor_rank",
            "yearmonth",
            "own_status",
            weight_col_name,
            "rank_weight",
            *_extra_norm_cols,
            *[
                F.col(c)
                for c in performance
            ],
            F.col(sales_cl),
        )
    )

    # --- Compute combined weight from norm_weights dict ---
    _cw_expr = F.lit(1.0)
    for _col_name, _exp in norm_weights.items():
        _cw_expr = _cw_expr * F.pow(
            F.greatest(F.col(_col_name), F.lit(1.0)), F.lit(_exp)
        )
    train_data = train_data.withColumn("combined_weight", _cw_expr)


    # ========================================================
    # Assign release-month index to each like game
    # ========================================================

    month_window = (
        Window
        .partitionBy(
            "target_game_name",
            "game_name",
        )
        .orderBy("yearmonth")
    )

    train_data = train_data.withColumn(
        "month_index",
        F.row_number().over(month_window),
    )


    # ========================================================
    # Remove games with no total sales
    #
    # This must happen BEFORE calculating the mean/std so that
    # the mean and standard deviation use the same games.
    # ========================================================

    total_sales_agg = (
        train_data
        .groupBy(
            "target_game_name",
            "game_name",
        )
        .agg(
            F.sum(
                F.col(sales_cl)
            ).alias("total_sales")
        )
    )

    valid_games = (
        total_sales_agg
        .filter(
            F.col("total_sales") > 0
        )
        .select(
            "target_game_name",
            "game_name",
        )
    )

    train_data = (
        train_data
        .join(
            valid_games,
            [
                "target_game_name",
                "game_name",
            ],
            "inner",
        )
    )


    # ========================================================
    # Calculate curve length
    # ========================================================

    game_lengths = (
        train_data
        .groupBy(
            "target_game_name",
            "game_name",
        )
        .agg(
            F.max("month_index")
            .alias("game_length")
        )
    )

    curve_lengths = (
        game_lengths
        .groupBy("target_game_name")
        .agg(
            F.min("game_length")
            .alias("curve_length")
        )
        .withColumn(
            "curve_length",
            F.least(
                F.col("curve_length"),
                F.lit(max_curve_length_batch),
            ),
        )
    )

    train_data = (
        train_data
        .join(
            curve_lengths,
            "target_game_name",
            "inner",
        )
        .filter(
            F.col("month_index")
            <= F.col("curve_length")
        )
    )


    # ========================================================
    # Calculate cumulative sales for EACH like game first
    #
    # This allows us to calculate the standard deviation of
    # cumulative curves instead of summing monthly std values.
    # ========================================================

    game_cumsum_window = (
        Window
        .partitionBy(
            "target_game_name",
            "game_name",
        )
        .orderBy("month_index")
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow,
        )
    )

    train_data = train_data.withColumn(
        f"cumsum_{sales_cl}",
        F.sum(
            F.col(sales_cl)
        ).over(game_cumsum_window),
    )


    # ========================================================
    # Weighted mean and weighted population std
    #
    # A separate valid-weight denominator is calculated for
    # every metric so null metric values do not contribute
    # weight to that metric.
    # ========================================================

    stats_cols = (
        performance
        + [
            sales_cl,
            f"cumsum_{sales_cl}",
            "months_since_release",
        ]
    )

    agg_exprs = []

    for c in stats_cols:

        valid_weight = F.when(
            F.col(c).isNotNull()
            & F.col("combined_weight").isNotNull()
            & (F.col("combined_weight") > 0),
            F.col("combined_weight"),
        ).otherwise(F.lit(0.0))

        weighted_sum = F.sum(
            F.when(
                F.col(c).isNotNull(),
                F.col(c) * valid_weight,
            ).otherwise(F.lit(0.0))
        )

        weighted_square_sum = F.sum(
            F.when(
                F.col(c).isNotNull(),
                F.col(c)
                * F.col(c)
                * valid_weight,
            ).otherwise(F.lit(0.0))
        )

        valid_weight_sum = F.sum(valid_weight)

        weighted_mean = (
            weighted_sum / valid_weight_sum
        )

        weighted_variance = (
            weighted_square_sum
            / valid_weight_sum
            - F.pow(weighted_mean, 2)
        )

        agg_exprs.extend(
            [
                F.when(
                    valid_weight_sum > 0,
                    weighted_mean,
                )
                .otherwise(
                    F.lit(None).cast("double")
                )
                .alias(c),

                F.when(
                    valid_weight_sum > 0,
                    F.sqrt(
                        F.greatest(
                            weighted_variance,
                            F.lit(0.0),
                        )
                    ),
                )
                .otherwise(
                    F.lit(None).cast("double")
                )
                .alias(f"{c}_std"),
            ]
        )

    agg_exprs.extend(
        [
            F.sum(
                F.when(
                    F.col("combined_weight") > 0,
                    F.col("combined_weight"),
                ).otherwise(F.lit(0.0))
            ).alias(
                "total_combined_weight"
            ),

            F.countDistinct(
                "game_name"
            ).alias(
                "n_like_games_used"
            ),

            F.sort_array(
                F.collect_set("game_name")
            ).alias(
                "like_games_used"
            ),
        ]
    )


    # ========================================================
    # Aggregate like games
    #
    # Do not group by own_status because doing so can create
    # multiple forecast rows for the same target/month.
    # ========================================================

    pred_curve = (
        train_data
        .groupBy(
            "target_game_name",
            "month_index",
            "curve_length",
        )
        .agg(*agg_exprs)
        .withColumns(
            {
                "run_id": F.lit(current_run_id),
                "dataset_split": F.lit(
                    current_split
                ),
                "experiment_name": F.lit(
                    current_exp_name
                ),
                "cosine_distance_threshold": F.lit(
                    cosine_distance_threshold
                ),
                "norm_weights": F.lit(
                    str(norm_weights)
                ),
            }
        )
    )


    # ========================================================
    # Create exact mean ± 2 weighted std bands
    #
    # Lower bounds are clipped at zero because sales and the
    # current performance metrics should not be negative.
    # ========================================================

    band_exprs = {}

    for c in stats_cols:

        band_exprs[f"{c}_upper"] = (
            F.col(c)
            + 2.0 * F.col(f"{c}_std")
        )

        band_exprs[f"{c}_lower"] = (
            F.greatest(
                F.col(c)
                - 2.0 * F.col(f"{c}_std"),
                F.lit(0.0),
            )
        )

    pred_curve = pred_curve.withColumns(
        band_exprs
    )


    # ========================================================
    # Stage 2: Fit cumulative-sales models
    # ========================================================

    curve_pd = (
        pred_curve
        .select(
            "target_game_name",
            "month_index",
            f"cumsum_{sales_cl}",
            f"cumsum_{sales_cl}_std",
        )
        .orderBy(
            "target_game_name",
            "month_index",
        )
        .toPandas()
    )

    forecast_rows = []
    model_rows = []

    for game, gdf in curve_pd.groupby(
        "target_game_name"
    ):

        gdf = (
            gdf
            .sort_values("month_index")
            .dropna(
                subset=[
                    f"cumsum_{sales_cl}",
                    f"cumsum_{sales_cl}_std",
                ]
            )
            .reset_index(drop=True)
        )

        if len(gdf) == 0:
            continue

        x = (
            gdf["month_index"]
            .to_numpy(dtype=float)
        )

        y = (
            gdf[f"cumsum_{sales_cl}"]
            .to_numpy(dtype=float)
        )

        cumulative_std = (
            gdf[f"cumsum_{sales_cl}_std"]
            .to_numpy(dtype=float)
        )

        best_r2 = -np.inf
        best_name = None
        best_pred = None
        best_func = None

        all_fits = {}
        all_fit_predictions = {}
        all_fit_rmse = {}
        all_fit_mae = {}

        fit_methods = [
            ("Linear", _fit_linear),
            ("Polynomial_deg2", _fit_poly2),
            ("Logarithmic", _fit_log),
        ]

        for name, fit_fn in fit_methods:

            coeffs, func = fit_fn(x, y)

            fitted_values = _clip_nonnegative(
                func(x)
            )

            score = _r_squared(
                y,
                fitted_values,
            )

            all_fits[name] = score
            all_fit_predictions[name] = (
                fitted_values
            )

            fit_errors = fitted_values - y

            all_fit_rmse[name] = np.sqrt(
                np.mean(fit_errors**2)
            )

            all_fit_mae[name] = np.mean(
                np.abs(fit_errors)
            )

            if score > best_r2:
                best_r2 = score
                best_name = name
                best_pred = fitted_values
                best_func = func

        model_rows.append(
            {
                "target_game_name": game,
                "dataset_split": current_split,
                "experiment_name": current_exp_name,
                "run_id": current_run_id,
                "cosine_distance_threshold": cosine_distance_threshold,
                "norm_weights": str(norm_weights),
                "best_model": best_name,
                "R2": round(best_r2, 4),
                "R2_linear": round(
                    all_fits["Linear"],
                    4,
                ),
                "RMSE_linear": round(
                    all_fit_rmse["Linear"],
                    4,
                ),
                "MAE_linear": round(
                    all_fit_mae["Linear"],
                    4,
                ),
                "R2_poly2": round(
                    all_fits["Polynomial_deg2"],
                    4,
                ),
                "RMSE_poly2": round(
                    all_fit_rmse[
                        "Polynomial_deg2"
                    ],
                    4,
                ),
                "MAE_poly2": round(
                    all_fit_mae[
                        "Polynomial_deg2"
                    ],
                    4,
                ),
                "R2_log": round(
                    all_fits["Logarithmic"],
                    4,
                ),
                "RMSE_log": round(
                    all_fit_rmse["Logarithmic"],
                    4,
                ),
                "MAE_log": round(
                    all_fit_mae["Logarithmic"],
                    4,
                ),
            }
        )

        # Use the same nonnegative fitted baseline that was
        # used to calculate R².
        fitted_baseline = _clip_nonnegative(
            best_func(x)
        )

        # Exact fitted baseline ± 2 weighted std.
        forecast_upper = (
            fitted_baseline
            + 2.0 * cumulative_std
        )

        forecast_lower = _clip_nonnegative(
            fitted_baseline
            - 2.0 * cumulative_std
        )

        for i, release_month in enumerate(
            x.astype(int)
        ):

            forecast_rows.append(
                {
                    "target_game_name": game,
                    "dataset_split": current_split,
                    "experiment_name": current_exp_name,
                    "run_id": current_run_id,
                    "cosine_distance_threshold": cosine_distance_threshold,
                    "norm_weights": str(norm_weights),
                    "release_month": int(
                        release_month
                    ),
                    "ForecastUnitBaseline": float(
                        fitted_baseline[i]
                    ),
                    "ForecastUnitBaseline_upper": float(
                        forecast_upper[i]
                    ),
                    "ForecastUnitBaseline_lower": float(
                        forecast_lower[i]
                    ),
                    "ForecastUnitBaseline_linear": float(
                        all_fit_predictions[
                            "Linear"
                        ][i]
                    ),
                    "ForecastUnitBaseline_poly2": float(
                        all_fit_predictions[
                            "Polynomial_deg2"
                        ][i]
                    ),
                    "ForecastUnitBaseline_log": float(
                        all_fit_predictions[
                            "Logarithmic"
                        ][i]
                    ),
                    "best_model": best_name,
                    "R2": round(best_r2, 4),
                }
            )


    forecast_iter = pd.DataFrame(
        forecast_rows
    )

    model_iter = pd.DataFrame(
        model_rows
    )

    if forecast_iter.empty:
        print(
            "    No valid forecast rows. Skipping."
        )
        continue


    # ========================================================
    # Stage 3: Add actuals, shift alignment, and errors
    # ========================================================

    target_games_iter = (
        forecast_iter["target_game_name"]
        .unique()
        .tolist()
    )

    actuals_iter = all_actuals_pd[
        all_actuals_pd["target_game_name"].isin(
            target_games_iter
        )
    ].copy()

    forecast_enriched = (
        forecast_iter
        .merge(
            actuals_iter,
            on=[
                "target_game_name",
                "release_month",
            ],
            how="left",
        )
    )


    # ========================================================
    # Add predicted performance metrics
    # ========================================================

    pred_perf_pd = (
        pred_curve
        .select(
            "target_game_name",
            F.col("month_index").alias(
                "release_month"
            ),
            *[
                F.col(c).alias(
                    f"predicted_{c}"
                )
                for c in performance
            ],
            F.col("months_since_release"),
        )
        .toPandas()
    )

    forecast_enriched = (
        forecast_enriched
        .merge(
            pred_perf_pd,
            on=[
                "target_game_name",
                "release_month",
            ],
            how="left",
        )
    )


    # ========================================================
    # Shift actuals to forecast starting month
    # ========================================================

    shifted_games = []

    for game, gdf in forecast_enriched.groupby(
        "target_game_name"
    ):

        gdf = (
            gdf
            .sort_values("release_month")
            .reset_index(drop=True)
        )

        actual_sale_col = (
            f"actual_{sales_cl}"
        )

        fc_nonzero = gdf[
            gdf["ForecastUnitBaseline"] > 0
        ]

        if len(fc_nonzero) > 0:
            first_fc_month = int(
                fc_nonzero[
                    "release_month"
                ].iloc[0]
            )
        else:
            first_fc_month = 1

        has_actual_sales = (
            actual_sale_col in gdf.columns
            and gdf[actual_sale_col]
            .notna()
            .any()
        )

        shift = 0

        if has_actual_sales:

            actual_nonzero = gdf[
                gdf[actual_sale_col].notna()
                & (gdf[actual_sale_col] > 0)
            ]

            if len(actual_nonzero) > 0:
                first_actual_month = int(
                    actual_nonzero[
                        "release_month"
                    ].iloc[0]
                )

                shift = (
                    first_fc_month
                    - first_actual_month
                )

        gdf["shift_months"] = shift

        game_actual_full = actuals_iter[
            actuals_iter["target_game_name"]
            == game
        ].copy()

        if (
            len(game_actual_full) > 0
            and shift != 0
        ):

            game_actual_full[
                "release_month"
            ] = (
                game_actual_full[
                    "release_month"
                ]
                + shift
            )

            shifted_sales = (
                game_actual_full[
                    [
                        "target_game_name",
                        "release_month",
                        actual_sale_col,
                    ]
                ]
                .rename(
                    columns={
                        actual_sale_col:
                            f"shifted_actual_{sales_cl}"
                    }
                )
            )

            gdf = gdf.merge(
                shifted_sales,
                on=[
                    "target_game_name",
                    "release_month",
                ],
                how="left",
            )

            if shift_flag_batch:

                performance_rename = {
                    f"actual_{c}":
                        f"shifted_actual_{c}"
                    for c in performance
                }

                shifted_performance = (
                    game_actual_full[
                        [
                            "target_game_name",
                            "release_month",
                        ]
                        + [
                            f"actual_{c}"
                            for c in performance
                        ]
                    ]
                    .rename(
                        columns=performance_rename
                    )
                )

                gdf = gdf.merge(
                    shifted_performance,
                    on=[
                        "target_game_name",
                        "release_month",
                    ],
                    how="left",
                )

            else:

                for c in performance:
                    gdf[
                        f"shifted_actual_{c}"
                    ] = gdf.get(
                        f"actual_{c}"
                    )

        else:

            gdf[
                f"shifted_actual_{sales_cl}"
            ] = gdf.get(actual_sale_col)

            for c in performance:
                gdf[
                    f"shifted_actual_{c}"
                ] = gdf.get(
                    f"actual_{c}"
                )

        shifted_games.append(gdf)


    forecast_final = pd.concat(
        shifted_games,
        ignore_index=True,
    )

    forecast_final = (
        forecast_final
        .sort_values(
            [
                "target_game_name",
                "release_month",
            ]
        )
        .reset_index(drop=True)
    )


    # ========================================================
    # Cumulative shifted actual sales
    # ========================================================

    forecast_final[
        f"shifted_actual_{sales_cl_cumsum}"
    ] = (
        forecast_final
        .groupby(
            "target_game_name",
            sort=False,
        )[f"shifted_actual_{sales_cl}"]
        .cumsum()
    )


    # ========================================================
    # Error metrics
    # ========================================================

    error_records = []

    for game, gdf in forecast_final.groupby(
        "target_game_name"
    ):

        rec = {
            "target_game_name": game
        }

        actual_cumulative_sales = gdf[
            f"shifted_actual_{sales_cl_cumsum}"
        ]

        forecast_curve_cols = {
            "linear": (
                "ForecastUnitBaseline_linear"
            ),
            "poly2": (
                "ForecastUnitBaseline_poly2"
            ),
            "log": (
                "ForecastUnitBaseline_log"
            ),
        }

        if _is_all_zero(
            actual_cumulative_sales
        ):

            rec["MAPE_cum_sales"] = np.nan
            rec["RMSE_cum_sales"] = np.nan
            rec["MAE_cum_sales"] = np.nan

            for curve_name in forecast_curve_cols:
                rec[
                    f"RMSE_forecast_{curve_name}"
                ] = np.nan
                rec[
                    f"MAE_forecast_{curve_name}"
                ] = np.nan

        else:

            valid_sales = gdf[
                actual_cumulative_sales.notna()
                & (actual_cumulative_sales != 0)
            ]

            if len(valid_sales) > 0:

                sales_errors = (
                    valid_sales[
                        "ForecastUnitBaseline"
                    ]
                    - valid_sales[
                        f"shifted_actual_"
                        f"{sales_cl_cumsum}"
                    ]
                )

                rec["MAPE_cum_sales"] = round(
                    (
                        sales_errors.abs()
                        / valid_sales[
                            f"shifted_actual_"
                            f"{sales_cl_cumsum}"
                        ].abs()
                    ).mean() * 100,
                    2,
                )

                rec["RMSE_cum_sales"] = round(
                    np.sqrt(
                        (
                            sales_errors**2
                        ).mean()
                    ),
                    4,
                )

                rec["MAE_cum_sales"] = round(
                    sales_errors.abs().mean(),
                    4,
                )

                for curve_name, curve_col in (
                    forecast_curve_cols.items()
                ):

                    curve_errors = (
                        valid_sales[curve_col]
                        - valid_sales[
                            f"shifted_actual_"
                            f"{sales_cl_cumsum}"
                        ]
                    )

                    rec[
                        f"RMSE_forecast_{curve_name}"
                    ] = round(
                        np.sqrt(
                            (
                                curve_errors**2
                            ).mean()
                        ),
                        4,
                    )

                    rec[
                        f"MAE_forecast_{curve_name}"
                    ] = round(
                        curve_errors.abs().mean(),
                        4,
                    )

            else:

                rec["MAPE_cum_sales"] = np.nan
                rec["RMSE_cum_sales"] = np.nan
                rec["MAE_cum_sales"] = np.nan

                for curve_name in forecast_curve_cols:
                    rec[
                        f"RMSE_forecast_{curve_name}"
                    ] = np.nan
                    rec[
                        f"MAE_forecast_{curve_name}"
                    ] = np.nan


        # Performance metric errors.
        for c in performance:

            actual_performance = gdf[
                f"shifted_actual_{c}"
            ]

            if _is_all_zero(
                actual_performance
            ):

                rec[f"MAPE_{c}"] = np.nan
                rec[f"RMSE_{c}"] = np.nan
                continue

            valid_performance = gdf[
                actual_performance.notna()
                & (actual_performance != 0)
            ]

            if len(valid_performance) > 0:

                performance_errors = (
                    valid_performance[
                        f"predicted_{c}"
                    ]
                    - valid_performance[
                        f"shifted_actual_{c}"
                    ]
                )

                rec[f"MAPE_{c}"] = round(
                    (
                        performance_errors.abs()
                        / valid_performance[
                            f"shifted_actual_{c}"
                        ].abs()
                    ).mean() * 100,
                    2,
                )

                rec[f"RMSE_{c}"] = round(
                    np.sqrt(
                        (
                            performance_errors**2
                        ).mean()
                    ),
                    4,
                )

            else:

                rec[f"MAPE_{c}"] = np.nan
                rec[f"RMSE_{c}"] = np.nan

        rec["shift_months"] = int(
            gdf["shift_months"].iloc[0]
        )

        error_records.append(rec)


    error_iter = pd.DataFrame(
        error_records
    )

    model_enriched = (
        model_iter
        .merge(
            error_iter,
            on="target_game_name",
            how="left",
        )
    )


    # ========================================================
    # Select output columns
    # ========================================================

    out_cols = [
        "target_game_name",
        "dataset_split",
        "release_month",
        "run_id",
        "experiment_name",
        "cosine_distance_threshold",
        "norm_weights",
        "months_since_release",
        "best_model",
        "R2",
        "ForecastUnitBaseline",
        "ForecastUnitBaseline_upper",
        "ForecastUnitBaseline_lower",
        "ForecastUnitBaseline_linear",
        "ForecastUnitBaseline_poly2",
        "ForecastUnitBaseline_log",
        f"shifted_actual_{sales_cl}",
        f"shifted_actual_{sales_cl_cumsum}",
        "shift_months",
    ]

    for c in performance:
        out_cols.extend(
            [
                f"predicted_{c}",
                f"shifted_actual_{c}",
            ]
        )

    out_cols = [
        c
        for c in out_cols
        if c in forecast_final.columns
    ]

    all_forecast_results.append(
        forecast_final[out_cols]
    )

    all_model_summaries.append(
        model_enriched
    )


# ============================================================
# Combine and write destination tables
# ============================================================

if not all_forecast_results:
    raise ValueError(
        "No forecast results were generated."
    )

if not all_model_summaries:
    raise ValueError(
        "No model summaries were generated."
    )

all_forecasts_combined = pd.concat(
    all_forecast_results,
    ignore_index=True,
)

all_models_combined = pd.concat(
    all_model_summaries,
    ignore_index=True,
)



In [0]:
display(all_forecasts_combined)
display(all_models_combined)

In [0]:

# ============================================================
# Save forecast results
# ============================================================

(
    spark
    .createDataFrame(all_forecasts_combined)
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(result_dest)
)


# ============================================================
# Save model performance results
# ============================================================

(
    spark
    .createDataFrame(all_models_combined)
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(performance_dest)
)


# ============================================================
# Completion summary
# ============================================================

print("=" * 60)

print(
    f"Saved {len(all_forecasts_combined)} "
    f"forecast rows → {result_dest}"
)

print(
    f"Saved {len(all_models_combined)} "
    f"model summary rows → {performance_dest}"
)

print(
    f"Run/split/experiment combinations processed: "
    f"{len(all_combos)}"
)

print(
    f"Shift flag: {shift_flag_batch}"
)
